<a href="https://colab.research.google.com/github/mloulaps-rgb/---/blob/main/HCI_%D0%9B%D0%A02_%D0%9D%D0%B0%D1%97%D0%B2%D0%BD%D0%B8%D0%B9_%D0%91%D0%B0%D1%94%D1%81_%D0%92%D0%B5%D0%BB%D0%B8%D1%87%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Лабораторна робота №2. Наївний баєсів класифікатор для аналізу тональності текстів
# Крок 1. Завантаження бібліотек і набору даних twitter_samples, поділ на вибірки.
import nltk
import numpy as np
import re
import string
from nltk.corpus import twitter_samples, stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

nltk.download('twitter_samples')
nltk.download('stopwords')

all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

# Поділ 80/20: по 4000 на навчання і по 1000 на тест для кожного класу
test_pos  = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg  = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg
test_x  = test_pos + test_neg

# Мітки: 1 - позитивний клас, 0 - негативний
train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y  = np.append(np.ones(len(test_pos)),  np.zeros(len(test_neg)))

print("Усього твітів для навчання:", len(train_x))
print("Усього твітів для тесту:", len(test_x))

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Усього твітів для навчання: 8000
Усього твітів для тесту: 2000


In [2]:
# Крок 1 (продовження). Попереднє оброблення тексту (process_tweet).
def process_tweet(tweet):
    """Очищує твіт і повертає список оброблених слів."""
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # прибираємо біржові тикери типу $GE
    tweet = re.sub(r'\$\w*', '', tweet)
    # прибираємо позначку ретвіту "RT"
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    # прибираємо гіперпосилання
    tweet = re.sub(r'https?://[^\s\n\r]+', '', tweet)
    # прибираємо символ хештегу #
    tweet = re.sub(r'#', '', tweet)
    # токенізація
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    tweet_tokens = tokenizer.tokenize(tweet)

    tweets_clean = []
    for word in tweet_tokens:
        if (word not in stopwords_english and      # видалення стоп-слів
                word not in string.punctuation):    # видалення пунктуації
            stem_word = stemmer.stem(word)          # стемінг
            tweets_clean.append(stem_word)
    return tweets_clean

# Демонстрація роботи
demo = train_x[0]
print("Початковий твіт:\n", demo)
print("\nПісля обробки:\n", process_tweet(demo))

Початковий твіт:
 #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)

Після обробки:
 ['followfriday', 'top', 'engag', 'member', 'commun', 'week', ':)']


In [3]:
# Крок 2. Побудова словника частотності слів для позитивного і негативного класів.
def count_tweets(result, tweets, ys):
    """Будує словник {(слово, мітка): кількість появ}."""
    for y, tweet in zip(ys, tweets):
        for word in process_tweet(tweet):
            pair = (word, y)
            result[pair] = result.get(pair, 0) + 1
    return result

freqs = count_tweets({}, train_x, train_y)
print("Кількість унікальних пар (слово, клас):", len(freqs))

# Приклади частот для кількох слів
for word in ['happi', 'sad', 'love', 'hate', 'good', 'bad', ':)', ':(']:
    pos = freqs.get((word, 1.0), 0)
    neg = freqs.get((word, 0.0), 0)
    print(f"  {word!r:10} -> позитив: {pos:5d}   негатив: {neg:5d}")

Кількість унікальних пар (слово, клас): 11397
  'happi'    -> позитив:   162   негатив:    18
  'sad'      -> позитив:     5   негатив:   100
  'love'     -> позитив:   336   негатив:   128
  'hate'     -> позитив:     9   негатив:    45
  'good'     -> позитив:   191   негатив:    83
  'bad'      -> позитив:    14   негатив:    54
  ':)'       -> позитив:  2960   негатив:     2
  ':('       -> позитив:     1   негатив:  3675


In [4]:
# Кроки 3-4. Навчання: логарифмічна апріорна ймовірність (log prior)
#            та логарифмічні правдоподібності (log likelihood) зі згладжуванням Лапласа.
def train_naive_bayes(freqs, train_x, train_y):
    loglikelihood = {}
    logprior = 0.0

    # словник унікальних слів
    vocab = set([pair[0] for pair in freqs.keys()])
    V = len(vocab)

    # загальна кількість позитивних і негативних слів (з урахуванням повторів)
    N_pos = N_neg = 0
    for pair in freqs.keys():
        if pair[1] > 0:
            N_pos += freqs[pair]
        else:
            N_neg += freqs[pair]

    # кількість документів (твітів) та документів кожного класу
    D = len(train_y)
    D_pos = int(np.sum(train_y == 1))
    D_neg = int(np.sum(train_y == 0))

    # логарифмічна апріорна ймовірність
    logprior = np.log(D_pos) - np.log(D_neg)

    # логарифмічна правдоподібність кожного слова
    for word in vocab:
        freq_pos = freqs.get((word, 1.0), 0)
        freq_neg = freqs.get((word, 0.0), 0)
        # ймовірності зі згладжуванням add-1 (Лапласа)
        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_neg + 1) / (N_neg + V)
        loglikelihood[word] = np.log(p_w_pos / p_w_neg)

    return logprior, loglikelihood

logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)
print("Логарифмічна апріорна ймовірність (logprior):", round(float(logprior), 6))
print("Розмір словника (V):", len(loglikelihood))

Логарифмічна апріорна ймовірність (logprior): 0.0
Розмір словника (V): 9143


In [5]:
# Кроки 5-6. Функція класифікатора та оцінка точності на тестовому наборі.
def naive_bayes_predict(tweet, logprior, loglikelihood):
    """Повертає сумарну оцінку: >0 - позитивний, <0 - негативний твіт."""
    words = process_tweet(tweet)
    p = logprior
    for word in words:
        if word in loglikelihood:
            p += loglikelihood[word]
    return p

def test_naive_bayes(test_x, test_y, logprior, loglikelihood):
    """Частка правильно класифікованих твітів."""
    y_hats = []
    for tweet in test_x:
        y_hat = 1 if naive_bayes_predict(tweet, logprior, loglikelihood) > 0 else 0
        y_hats.append(y_hat)
    error = np.mean(np.abs(np.array(y_hats) - test_y))
    accuracy = 1 - error
    return accuracy

accuracy = test_naive_bayes(test_x, test_y, logprior, loglikelihood)
print(f"Точність наївного баєсового класифікатора на тесті: {accuracy:.4f} ({accuracy*100:.2f}%)")

Точність наївного баєсового класифікатора на тесті: 0.9955 (99.55%)


In [6]:
# Крок 9. Перевірка класифікатора на власних твітах.
for my_tweet in ["I am happy and love this great day :)",
                 "This is the worst, I hate it, so sad :(",
                 "She smiled at the ridiculous movie and great song"]:
    p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
    label = "Позитивна тональність" if p > 0 else "Негативна тональність"
    print(f"'{my_tweet}'\n  оцінка = {p:.4f} -> {label}\n")

'I am happy and love this great day :)'
  оцінка = 12.5554 -> Позитивна тональність

'This is the worst, I hate it, so sad :('
  оцінка = -12.9349 -> Негативна тональність

'She smiled at the ridiculous movie and great song'
  оцінка = 1.8257 -> Позитивна тональність



In [7]:
# Крок 7. Аналіз найбільш "позитивних" та "негативних" слів за співвідношенням частот.
def lookup(freqs, word, label):
    return freqs.get((word, label), 0)

def get_ratio(freqs, word):
    """Співвідношення (pos+1)/(neg+1) для слова."""
    pos = lookup(freqs, word, 1.0)
    neg = lookup(freqs, word, 0.0)
    return {'positive': pos, 'negative': neg, 'ratio': (pos + 1) / (neg + 1)}

vocab = list(set([pair[0] for pair in freqs.keys()]))
ratios = {w: get_ratio(freqs, w) for w in vocab}

# топ-10 найбільш позитивних слів (найбільше ratio)
top_pos = sorted(ratios.items(), key=lambda kv: kv[1]['ratio'], reverse=True)[:10]
# топ-10 найбільш негативних слів (найменше ratio)
top_neg = sorted(ratios.items(), key=lambda kv: kv[1]['ratio'])[:10]

print("Топ-10 найбільш ПОЗИТИВНИХ слів (pos / neg / ratio):")
for w, d in top_pos:
    print(f"  {w!r:12} {d['positive']:5d} {d['negative']:5d}  ratio={d['ratio']:.1f}")

print("\nТоп-10 найбільш НЕГАТИВНИХ слів (pos / neg / ratio):")
for w, d in top_neg:
    print(f"  {w!r:12} {d['positive']:5d} {d['negative']:5d}  ratio={d['ratio']:.3f}")

Топ-10 найбільш ПОЗИТИВНИХ слів (pos / neg / ratio):
  ':)'          2960     2  ratio=987.0
  ':-)'          552     0  ratio=553.0
  ':d'           523     0  ratio=524.0
  ':p'           105     0  ratio=106.0
  'stat'          51     0  ratio=52.0
  'warsaw'        44     0  ratio=45.0
  'bam'           44     0  ratio=45.0
  'blog'          27     0  ratio=28.0
  'fback'         26     0  ratio=27.0
  'followfriday'    23     0  ratio=24.0

Топ-10 найбільш НЕГАТИВНИХ слів (pos / neg / ratio):
  ':('             1  3675  ratio=0.001
  ':-('            0   386  ratio=0.003
  '♛'              0   210  ratio=0.005
  '》'              0   210  ratio=0.005
  '>:('            0    43  ratio=0.023
  'wi̇ll'          0    35  ratio=0.028
  'ｍｅ'             0    35  ratio=0.028
  'ｓｅｅ'            0    35  ratio=0.028
  'beli̇ev'        0    35  ratio=0.028
  'justi̇n'        0    35  ratio=0.028


In [8]:
# Крок 8. Аналіз помилок класифікації (хибно класифіковані твіти).
print("Твіти, на яких класифікатор помилився:\n")
errors = 0
for tweet, y in zip(test_x, test_y):
    y_hat = 1 if naive_bayes_predict(tweet, logprior, loglikelihood) > 0 else 0
    if y_hat != int(y):
        errors += 1
        if errors <= 10:   # показуємо перші 10 помилок
            print(f"  істинна={int(y)}  прогноз={y_hat}  |  {' '.join(process_tweet(tweet))}")
print(f"\nУсього помилок на тесті: {errors} з {len(test_x)}")

Твіти, на яких класифікатор помилився:

  істинна=1  прогноз=0  |  truli later move know queen bee upward bound movingonup
  істинна=1  прогноз=0  |  new report talk burn calori cold work harder warm feel better weather :p
  істинна=1  прогноз=0  |  harri niall 94 harri born ik stupid wanna chang :d
  істинна=1  прогноз=0  |  park get sunlight
  істинна=1  прогноз=0  |  uff itna miss karhi thi ap :p
  істинна=0  прогноз=1  |  hello info possibl interest jonatha close join beti :( great
  істинна=0  прогноз=1  |  u prob fun david
  істинна=0  прогноз=1  |  pat jay
  істинна=0  прогноз=1  |  sr financi analyst expedia inc bellevu wa financ expediajob job job hire

Усього помилок на тесті: 9 з 2000
